# Baselines + Zero-Shot Qwen 2.5 7B — Validation Evaluation

Runs all three baselines on the validation set (726 examples) and saves results for the final comparison.

| Baseline | Method | Time |
|----------|--------|------|
| BM25 | Keyword overlap, no GPU | < 1 min |
| Sentence-Transformer | `all-mpnet-base-v2` cosine sim, CPU | ~3 min |
| Zero-shot Qwen 2.5 7B | Untuned model, same prompt | ~1.5 hr (T4) / ~25 min (A100) |

**Ground truth:** teacher scores from DeepSeek (the `score` field in each validation example)

## 0. Check GPU

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Install

In [ ]:
%%capture
!pip install rank_bm25 sentence-transformers scipy
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
!pip install --no-deps 'xformers<0.0.27' 'trl<0.9.0' peft accelerate bitsandbytes

In [ ]:
import json, os, time, collections
import numpy as np
import torch
from scipy.stats import pearsonr, spearmanr
print('Imports OK')

## 2. Load Validation Data

In [ ]:
!git clone https://github.com/eka026/fit-my-resume.git /content/repo
VAL_PATH = '/content/repo/data/instruction_tuning/instruction_tuning_validation.jsonl'

In [ ]:
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

val_data = load_jsonl(VAL_PATH)
print(f'Validation examples: {len(val_data)}')

# Parse out the fields we need
examples = []
for ex in val_data:
    parts  = ex['input'].split('JOB_DESCRIPTION:')
    resume = parts[0].replace('RESUME:', '').strip()
    job    = parts[1].strip()
    examples.append({
        'pair_id':       ex['metadata']['pair_id'],
        'strategy':      ex['metadata']['pairing_strategy'],
        'resume':        resume,
        'job':           job,
        'input':         ex['input'],
        'teacher_score': json.loads(ex['output'])['score'],
    })

teacher_scores = [ex['teacher_score'] for ex in examples]
print(f'Teacher score — mean: {np.mean(teacher_scores):.1f}, range: {min(teacher_scores)}–{max(teacher_scores)}')

## 3. Baseline 1 — BM25

Keyword overlap between the resume and job description. No GPU, runs in seconds.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

corpus = [tokenize(ex['resume']) for ex in examples]
bm25   = BM25Okapi(corpus)

bm25_scores = []
for i, ex in enumerate(examples):
    scores = bm25.get_scores(tokenize(ex['job']))
    bm25_scores.append(float(scores[i]))

pearson_bm25,  _ = pearsonr(teacher_scores, bm25_scores)
spearman_bm25, _ = spearmanr(teacher_scores, bm25_scores)
print(f'BM25 — Pearson: {pearson_bm25:.3f}  Spearman: {spearman_bm25:.3f}')

## 4. Baseline 2 — Sentence-Transformer

Cosine similarity of `all-mpnet-base-v2` embeddings, scaled to 0–100. Runs on CPU to keep GPU free for Qwen.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

st_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device='cpu')

print('Encoding resumes...')
resume_embs = st_model.encode([ex['resume'] for ex in examples], batch_size=32, show_progress_bar=True)
print('Encoding job descriptions...')
job_embs    = st_model.encode([ex['job']    for ex in examples], batch_size=32, show_progress_bar=True)

st_scores = [
    float(max(0.0, min(100.0, cosine_similarity(resume_embs[i:i+1], job_embs[i:i+1])[0][0] * 100)))
    for i in range(len(examples))
]

pearson_st,  _ = pearsonr(teacher_scores, st_scores)
spearman_st, _ = spearmanr(teacher_scores, st_scores)
mae_st = np.mean(np.abs(np.array(teacher_scores) - np.array(st_scores)))
print(f'Sentence-Transformer — Pearson: {pearson_st:.3f}  Spearman: {spearman_st:.3f}  MAE: {mae_st:.1f}')

## 5. Load Qwen 2.5 7B (4-bit, untuned)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print('Model loaded!')

## 6. Zero-Shot Inference on Full Validation Set

In [ ]:
SYSTEM_PROMPT = (
    'You are a professional resume evaluation assistant. '
    'Evaluate the resume against the job description and return structured JSON feedback. '
    'Do not invent or fabricate any experience not present in the resume.'
)

def run_inference(input_text, max_new_tokens=512):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': input_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids      = inputs,
            max_new_tokens  = max_new_tokens,
            temperature     = 0.1,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)

In [ ]:
qwen_results = []
parse_errors = 0
start = time.time()

for i, ex in enumerate(examples):
    response = run_inference(ex['input'])
    try:
        pred_score = json.loads(response)['score']
    except (json.JSONDecodeError, KeyError):
        pred_score = None
        parse_errors += 1

    qwen_results.append({
        'pair_id':    ex['pair_id'],
        'strategy':   ex['strategy'],
        'gt_score':   ex['teacher_score'],
        'pred_score': pred_score,
        'raw_output': response,
    })

    if (i + 1) % 50 == 0 or i == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (len(examples) - i - 1) / rate / 60
        print(f'[{i+1}/{len(examples)}]  {elapsed/60:.1f} min elapsed  ~{remaining:.0f} min left  '
              f'parse ok: {i+1-parse_errors}/{i+1}')

total_min = (time.time() - start) / 60
print(f'\nDone in {total_min:.1f} min  |  Parse errors: {parse_errors}/{len(examples)}')

## 7. Save Results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT_DIR = '/content/drive/MyDrive/fit-my-resume/results'
os.makedirs(OUT_DIR, exist_ok=True)

# Save zero-shot Qwen predictions
QWEN_PATH = f'{OUT_DIR}/zero_shot_qwen_val_outputs.jsonl'
with open(QWEN_PATH, 'w', encoding='utf-8') as f:
    for row in qwen_results:
        f.write(json.dumps(row) + '\n')
print(f'Saved Qwen outputs: {QWEN_PATH}')

# Also download as backup
LOCAL_PATH = '/content/zero_shot_qwen_val_outputs.jsonl'
with open(LOCAL_PATH, 'w', encoding='utf-8') as f:
    for row in qwen_results:
        f.write(json.dumps(row) + '\n')
from google.colab import files
files.download(LOCAL_PATH)

## 8. Results — All Baselines

In [ ]:
# Zero-shot Qwen metrics
valid_qwen = [(r['gt_score'], r['pred_score']) for r in qwen_results if r['pred_score'] is not None]
qwen_gt   = [v[0] for v in valid_qwen]
qwen_pred = [v[1] for v in valid_qwen]
pearson_qwen,  _ = pearsonr(qwen_gt, qwen_pred)
spearman_qwen, _ = spearmanr(qwen_gt, qwen_pred)
mae_qwen = np.mean(np.abs(np.array(qwen_gt) - np.array(qwen_pred)))
parse_rate = 100 * len(valid_qwen) / len(qwen_results)

print('=' * 68)
print(f'{"Baseline":<26} {"Pearson":>9} {"Spearman":>10} {"MAE":>7} {"Parse%":>8}')
print('-' * 68)
print(f'{"BM25":<26} {pearson_bm25:>9.3f} {spearman_bm25:>10.3f} {"—":>7} {"—":>8}')
print(f'{"Sentence-Transformer":<26} {pearson_st:>9.3f} {spearman_st:>10.3f} {mae_st:>7.1f} {"—":>8}')
print(f'{"Zero-shot Qwen 2.5 7B":<26} {pearson_qwen:>9.3f} {spearman_qwen:>10.3f} {mae_qwen:>7.1f} {parse_rate:>7.1f}%')
print('=' * 68)
print('(Fine-tuned Qwen row will be added by Enes after Aliya shares results)')

In [ ]:
# Per-strategy breakdown for zero-shot Qwen
strat_groups = collections.defaultdict(list)
for r in qwen_results:
    if r['pred_score'] is not None:
        strat_groups[r['strategy']].append((r['gt_score'], r['pred_score']))

print('Zero-shot Qwen — per pairing strategy:')
print(f'{"Strategy":<22} {"n":>5} {"MAE":>8} {"Spearman":>10}')
print('-' * 50)
for strat, pairs in sorted(strat_groups.items()):
    g = [p[0] for p in pairs]
    p = [p[1] for p in pairs]
    spr, _ = spearmanr(g, p)
    m = np.mean(np.abs(np.array(g) - np.array(p)))
    print(f'{strat:<22} {len(pairs):>5} {m:>8.1f} {spr:>10.3f}')

In [ ]:
# Sanity check — sample predictions
print(f'{"pair_id":<45} {"GT":>4} {"Pred":>6}')
print('-' * 60)
for r in qwen_results[:15]:
    pred_str = str(r['pred_score']) if r['pred_score'] is not None else 'ERR'
    print(f'{r["pair_id"][:44]:<45} {r["gt_score"]:>4} {pred_str:>6}')